# Iktos 3D Dock API - Usage Guide

This notebook shows how to run the Iktos 3D Dock API to dock a list of SMILES into a protein using a known reference ligand.

**Workflow overview:**
1. Initialize the API client
2. Check API quota
3. *(Optional)* Clean the protein PDB via the API
4. Prepare inputs (protein + reference ligand + list of SMILES)
5. Submit a single prediction request
6. Monitor progress
7. Retrieve results into a dataframe

## 1. Setup

In [ ]:
import os

import pandas as pd

from rdkit import Chem

from dock_3d_client import (
    Dock3DClient,
    ProteinInput,
    ReferenceLigandInput,
    LigandInput,
    load_pdb_file,
    load_molblock_file,
)

In [ ]:
API_URL = os.getenv("API_URL", "3D_DOCK_API_URL")
API_KEY = os.getenv("API_KEY", "YOUR_API_KEY")

client = Dock3DClient(API_URL, api_key=API_KEY)

## 2. Health Check & Quota

In [ ]:
health = client.health_check()
print(f"API Status: {health['status']}")

In [ ]:
quota = client.get_quota()
print(f"Quota: {quota.quota_used}/{quota.quota_max} used, {quota.quota_remaining} remaining")

## 3. Clean Protein PDB *(optional)*

Skip if you already have a cleaned PDB.

This is not a full preparation pipeline, but a targeted cleaning step to prevent common issues. It:

- Normalizes atom and residue names to standard PDB conventions
- Converts non-standard amino acids to their canonical equivalents (e.g., SEP → SER)
- Removes terminal cap residues (ACE / NME)
- Removes all non-residue molecules (e.g., water, ions), while retaining cofactors
- Resolves alternate atom locations by keeping only the highest-occupancy conformation

In [ ]:
PROTEIN_PDB_PATH = "path/to/protein_pdb.pdb"
CLEAN_PDB_PATH = "path/to/clean_pdb.pdb"

raw_pdb = load_pdb_file(PROTEIN_PDB_PATH)
clean_pdb = client.clean_protein(raw_pdb)
os.makedirs(os.path.dirname(CLEAN_PDB_PATH) or ".", exist_ok=True)
with open(CLEAN_PDB_PATH, "w") as f:
    f.write(clean_pdb)
print("Protein cleaned and saved.")


## 4. Prepare Inputs

A prediction job requires:
- **One protein** (`ProteinInput`): id + PDB mol_block
- **One reference ligand** (`ReferenceLigandInput`): id + 3D molblock (SDF/MOL format)
- **A list of query ligands** (`LigandInput`): each with an id and a SMILES string

In [ ]:
REFERENCE_LIGAND_PATH = "path/to/mol_template.sdf"
CLEAN_PDB_PATH = "path/to/clean_pdb.pdb"

# Load protein and reference ligand
protein = ProteinInput(
    id="my_target",
    mol_block=load_pdb_file(CLEAN_PDB_PATH),
)

reference_ligand = ReferenceLigandInput(
    id="reference_ligand",
    mol_block=load_molblock_file(REFERENCE_LIGAND_PATH),
)

print(f"Protein: {protein.id}  |  Reference ligand: {reference_ligand.id}")

In [ ]:
# List of query SMILES to dock against the protein
smiles_list = [
    "CCc1nn(CCO)c(CC)c1Oc1cc(C#N)cc(C#N)c1",
    "Cn1c(C#N)ccc1-c1ccc2c(c1)C(C)(C)OC(=S)N2",
    "Cc1csc(Nc2ncccc2Oc2ccccc2C#N)n1",
    # ... add as many SMILES as you like
]

ligands = [
    LigandInput(id=f"ligand_{i}", smiles=smi)
    for i, smi in enumerate(smiles_list)
]
print(f"Prepared {len(ligands)} ligands.")

## 5. Submit a Prediction Request


In [ ]:
response = client.create_request(
    protein=protein,
    reference_ligand=reference_ligand,
    ligands=ligands,
    batch_job_id="example_multi_smiles",
)

request_id = response["request_id"]
print(f"Request submitted. request_id: {request_id}")
print(f"Total ligands: {response['total_ligands']}")

## 6. Monitor Progress

You can either poll manually or use the built-in `wait_for_completion()` method. `wait_for_completion()` polls the request every `check_interval` seconds until all ligand jobs are in a terminal state. The optional `progress_callback` lets you print intermediate progress.

In [ ]:
# Manual status check
status = client.get_request_status(request_id)
summary = status.status_summary

print(f"Status: {status.status.value}")
print(f"Total:     {summary.total}")
print(f"Completed: {summary.completed} ({summary.completed_percent:.1f}%)")
print(f"Failed:    {summary.failed} ({summary.failed_percent:.1f}%)")
print(f"Pending:   {summary.pending + + summary.processing} ({(summary.processing_percent + summary.pending_percent):.1f}%)")

In [ ]:
def show_progress(request):
    s = request.status_summary
    if s is not None:
        print(
            f"  {s.completed}/{s.total} completed "
            f"({s.completed_percent:.1f}%)  |  failed: {s.failed}"
        )


final_status = client.wait_for_completion(
    request_id,
    check_interval=5,
    progress_callback=show_progress,
)

print(f"\nDone! Final status: {final_status.status.value}")

## 7. Retrieve Results

Each completed job returns:
- `vina_score`
- `pharmaco_score`
- `shape_score`
- `ligand_smiles`
- `ligand_pose_sdf` (predicted 3D pose)

In [ ]:
results = client.get_results(request_id)

rows = []
for ligand_id, res in results.items():
    row = {"ligand_id": ligand_id, "status": res["status"], "time_elapsed": res.get("time_elapsed")}
    if res["status"] == "COMPLETED":
        data = res["data"]
        row.update({
            "ligand_smiles": data.get("ligand_smiles"),
            "vina_score": data.get("vina_score"),
            "pharmaco_score": data.get("pharmaco_score"),
            "shape_score": data.get("shape_score"),
            "ligand_pose_sdf": data.get("ligand_pose_sdf"),
        })
    elif res["status"] == "FAILED":
        row["error"] = res.get("error")
    rows.append(row)

df = pd.DataFrame(rows)
df[[c for c in df.columns if c != "ligand_pose_sdf"]]

### Save predicted poses *(optional)*

In [ ]:
OUTPUT_DIR = "docked_poses"
OUTPUT_FILE = "docked_poses.sdf"
os.makedirs(OUTPUT_DIR, exist_ok=True)

sorted_df = df[df["ligand_pose_sdf"].notna()].copy()
sorted_df["_id_num"] = sorted_df["ligand_id"].str.extract(r"(\d+)$").astype(int)
sorted_df = sorted_df.sort_values("_id_num")

output_path = os.path.join(OUTPUT_DIR, OUTPUT_FILE)
with open(output_path, "w") as f:
    writer = Chem.SDWriter(f)
    for i, row in sorted_df.iterrows():
        mol = Chem.MolFromMolBlock(row["ligand_pose_sdf"], removeHs=False)
        if mol is None:
            continue
        mol.SetProp("_Name", str(row.get("ligand_id", f"ligand_{i}")))
        writer.write(mol)
    writer.close()

print(f"Saved {len(sorted_df)} poses to {output_path}")